In [ ]:
!pip install --upgrade --quiet pinecone pinecone-text pinecone-notebooks

In [6]:
from dotenv import load_dotenv
import os
import pinecone

load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")


In [7]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [9]:
from pinecone import Pinecone, ServerlessSpec
index_name = "hybrid-search-langchain-pinecone"
pc = Pinecone(api_key=pinecone_api_key) # Initialize Pinecone client

#create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="dotproduct", # sparse values supported for dotproduct metric
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

In [10]:
index = pc.index(index_name)
index

Index(host='https://hybrid-search-langchain-pinecone-ivw4duj.svc.aped-4627-b74a.pinecone.io')

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

c:\Users\KRISHNA\miniconda3\envs\mlenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2613.31it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [15]:
from pinecone_text.sparse import BM25Encoder
bm25_encoder = BM25Encoder().default()
bm25_encoder

In [16]:
sentences = [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
]

bm25_encoder.fit(sentences) #tfidf vectorizer is fitted on the sentences to learn the vocabulary and idf values

bm25_encoder.dump("bm25_encoder.json") #store the fitted encoder to a json file for later use

bm25_encoder = BM25Encoder().load("bm25_encoder.json") #load the fitted encoder from the json file

100%|██████████| 3/3 [00:00<00:00, 19.13it/s]


In [18]:
retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25_encoder, index=index)

In [19]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001AC2466F6D0>, index=Index(host='https://hybrid-search-langchain-pinecone-ivw4duj.svc.aped-4627-b74a.pinecone.io'))

In [23]:
retriever.add_texts(
    texts=sentences
)

  0%|          | 0/1 [00:00<?, ?it/s]


TypeError: Index.upsert() takes 1 positional argument but 2 positional arguments (and 1 keyword-only argument) were given